In [ ]:
# Imports

import helpers.helper_functions as hf
import mne
import os.path as op
from mne.channels import combine_channels
import pandas as pd
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import hilbert


ss = hf.settings_dict()

In [ ]:
# Important variables
tmin = 0.7
tmax = 3.7

In [ ]:
for subject_index in ss['subject_idx_list']:
    # loop over each event type
    for event_name in ss['event_id_list']:
        subjects_dir = ss['fs_subjects_dir']
        subject = ss['subject_id_list'][subject_index]
        print("loading dataset for subject: ", subject)

        save_dir = Path(ss['stc_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)

        stc_file = Path(ss['stc_dir']) / subject / event_name / f"{subject}-event-{event_name}-vol.stc"

        stc = mne.read_source_estimate(stc_file)
        cropped_stc = stc.copy().crop(tmin=tmin, tmax=tmax)

        # hilbert transform the stc

        analytic_signal = hilbert(cropped_stc.data, axis=1)
        phase = np.angle(analytic_signal).astype(float)
        amplitude = np.abs(analytic_signal)

        mean_amp = np.mean(amplitude, axis=1)

        # pick the voxel with the highest mean amplitude to be used as a reference instead of the retina one
        ref_voxel = np.argmax(mean_amp)

        # save the hilbert transformations in a new stc

        hil_stc = cropped_stc.copy()
        hil_stc.data = phase

        hil_stc.save(save_dir / f"{subject}-event-{event_name}-vol.stc" , overwrite=True)

        #TODO: add masking based on the amplitude of the signals



